In [ ]:
#!gdown 1MjDptEqh7mYAm1-CCIcrpibSl3fdNJyW
!gdown 1vFnniAePXroS1w7WSlWxCdFrpq_00Tvh
!gdown 1LgrUbgjYsfcnjf0x3D8aWOi-ZpGzbPB69jNW9PJ_68c
!gdown 1dI_bOfc2G03PQwcYAIzdxSyWNtR4DFBy

import tqdm
import math
import torch
from matplotlib import pyplot as plt
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from tqdm import tqdm
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import BayesianRidge
import copy
from sklearn.model_selection import KFold

FEATURES = ["Geothermal Header Hot Water Temperature", "Ambient Temperature", "Hot Water Supply Flow Rate"]
FEATURES_COL = ["TEMPERATURA_AMBIENTE___C_", "T____C_", "Potencial_Agua_promd__kg_s_", "rho___kg_m3_"]
N_MC = 5000
kgs_TO_GPM = 60.0 / 0.003785411784
TARGET_COL_USA = "Gross Power Output (Veris)"
TARGET_COL_COL = "W_exergía__kW_"
#TARGET_COL_COL = "W_nth__kW_"

GLOBAL_RANDOM_SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Downloading...
From: https://drive.google.com/uc?id=1vFnniAePXroS1w7WSlWxCdFrpq_00Tvh
To: /content/Green Machine Florida Canyon Hourly Data.xlsx
100% 204k/204k [00:00<00:00, 89.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LgrUbgjYsfcnjf0x3D8aWOi-ZpGzbPB69jNW9PJ_68c
From (redirected): https://docs.google.com/spreadsheets/d/1LgrUbgjYsfcnjf0x3D8aWOi-ZpGzbPB69jNW9PJ_68c/export?format=xlsx
To: /content/Dataset_Colombia_potencial_generacion.xlsx
148kB [00:00, 722kB/s]
Downloading...
From: https://drive.google.com/uc?id=1dI_bOfc2G03PQwcYAIzdxSyWNtR4DFBy
To: /content/potencial_generacion (1).csv
100% 372k/372k [00:00<00:00, 65.9MB/s]


# **Preprocesamiento base de datos USA (train)**

In [ ]:
df = pd.read_excel('/content/Green Machine Florida Canyon Hourly Data.xlsx')
FEATURES_USA = FEATURES + [TARGET_COL_USA]
df = df[FEATURES_USA]
df = df.drop(0)

# Fahrenheit a celcius
def fahrenheit_a_celsius_col(df, col_f, nueva_col=None):
    df[col_f] = (df[col_f] - 32) * 5 / 9
    return df

fahrenheit_a_celsius_col(df, ["Geothermal Header Hot Water Temperature", "Ambient Temperature"])
df

,Geothermal Header Hot Water Temperature,Ambient Temperature,Hot Water Supply Flow Rate,Gross Power Output (Veris)
1,106.813117,7.506019,161.4125,57.264639
2,106.897222,7.951698,163.290833,57.536389
3,106.926698,10.019753,163.722778,56.865861
4,107.101852,11.908488,147.834722,49.966861
5,107.18179,11.342438,160.743333,56.418611
...,...,...,...,...
1307,102.872191,19.836454,136.676404,50.115084
1308,102.368659,20.126206,136.481232,48.436162
1309,102.040895,20.278549,135.466944,52.757972
1310,95.495512,22.148406,114.419777,43.660724


# **Preprocesamiento base de datos Colombia (test)**

In [ ]:
df_2 = pd.read_csv("/content/potencial_generacion (1).csv")
features_col = FEATURES_COL + [TARGET_COL_COL]
df_2 = df_2[features_col]
df_2

,TEMPERATURA_AMBIENTE___C_,T____C_,Potencial_Agua_promd__kg_s_,rho___kg_m3_,W_exergía__kW_
0,26.49630,115.556372,0.000000,984.985238,0.000000
1,26.29722,113.958572,0.381690,967.980447,8.463874
2,26.38017,113.703853,0.105676,947.895970,4.809997
3,26.33593,113.011865,0.000000,978.574574,0.000000
4,26.47418,112.167403,0.005321,990.344388,0.011926
...,...,...,...,...,...
1209,25.57832,36.575098,0.225373,993.439920,0.191869
1210,23.06217,35.809347,0.000000,993.710308,0.000000
1211,24.17923,33.410556,2.315067,994.526997,1.408083
1212,24.95343,32.284969,1.507404,994.894005,0.584444


In [ ]:
df_filtrado = df_2[df_2["W_exergía__kW_"] != 0]
df_filtrado.value_counts("W_exergía__kW_")
df_filtrado

,TEMPERATURA_AMBIENTE___C_,T____C_,Potencial_Agua_promd__kg_s_,rho___kg_m3_,W_exergía__kW_
1,26.29722,113.958572,0.381690,967.980447,8.463874
2,26.38017,113.703853,0.105676,947.895970,4.809997
4,26.47418,112.167403,0.005321,990.344388,0.011926
5,26.21980,111.230930,0.336992,960.969384,10.125163
6,26.47418,110.913344,0.001434,987.422760,0.005910
...,...,...,...,...,...
1206,23.55987,37.288504,1.665291,993.183882,2.197887
1208,24.96449,36.905798,1.717899,993.321726,1.720381
1209,25.57832,36.575098,0.225373,993.439920,0.191869
1211,24.17923,33.410556,2.315067,994.526997,1.408083


In [ ]:
# Flujo másico a flujo volumétrico en gpb
def fm_to_fv(df, col_rho, col_fm, nueva_col=None):
    df[nueva_col] = (df[col_fm] * kgs_TO_GPM) /df[col_rho]
    return df

fm_to_fv(df_2, "rho___kg_m3_", "Potencial_Agua_promd__kg_s_", "Potencial_Agua_promd_gpm")
df_2.drop(columns=["rho___kg_m3_", "Potencial_Agua_promd__kg_s_"], inplace=True)
df_2["W_exergía__kW_"] = df_2["W_exergía__kW_"] * 0.09
df_2

,TEMPERATURA_AMBIENTE___C_,T____C_,W_exergía__kW_,Potencial_Agua_promd_gpm
0,26.49630,115.556372,0.000000,0.000000
1,26.29722,113.958572,0.761749,6.250030
2,26.38017,113.703853,0.432900,1.767066
3,26.33593,113.011865,0.000000,0.000000
4,26.47418,112.167403,0.001073,0.085156
...,...,...,...,...
1209,25.57832,36.575098,0.017268,3.595823
1210,23.06217,35.809347,0.000000,0.000000
1211,24.17923,33.410556,0.126727,36.896497
1212,24.95343,32.284969,0.052600,24.015471


In [ ]:
df_filtrado = df_2[df_2['W_exergía__kW_'] != 0]
df_filtrado.info()

df_filtrado= df_filtrado.dropna()
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 615 entries, 1 to 1212
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   TEMPERATURA_AMBIENTE___C_  569 non-null    float64
 1   T____C_                    615 non-null    float64
 2   W_exergía__kW_             615 non-null    float64
 3   Potencial_Agua_promd_gpm   615 non-null    float64
dtypes: float64(4)
memory usage: 24.0 KB
<class 'pandas.core.frame.DataFrame'>
Index: 569 entries, 1 to 1212
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   TEMPERATURA_AMBIENTE___C_  569 non-null    float64
 1   T____C_                    569 non-null    float64
 2   W_exergía__kW_             569 non-null    float64
 3   Potencial_Agua_promd_gpm   569 non-null    float64
dtypes: float64(4)
memory usage: 22.2 KB


# Organización de los conjuntos de train y test

In [ ]:
# Conjunto de entrenamiento
X_train = df[[
    "Geothermal Header Hot Water Temperature",
    "Ambient Temperature",
    "Hot Water Supply Flow Rate"
]].to_numpy()

y_train = df[TARGET_COL_USA].copy()

In [ ]:
#Conjunto de test
X_test = df_filtrado[[
    "T____C_",
    "TEMPERATURA_AMBIENTE___C_",
    "Potencial_Agua_promd_gpm"
]].to_numpy()

y_test = df_filtrado[TARGET_COL_COL].copy()

In [ ]:
# Escalado de variables
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# **1. Regresor bayesiano**

In [ ]:
model = BayesianRidge(compute_score=True, max_iter=3000)
model.fit(X_train_scaled, y_train)

y_pred_mean, y_pred_std = model.predict(X_test_scaled, return_std=True)

In [ ]:
mae = mean_absolute_error(y_test, y_pred_mean)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_mean))
r2 = r2_score(y_test, y_pred_mean)

print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"R²   = {r2:.4f}")

MAE  = 27.7325
RMSE = 29.5373
R²   = -20.3357


In [ ]:
import numpy as np
import plotly.graph_objects as go

idx = np.arange(len(y_test))
lower = y_pred_mean - y_pred_std
upper = y_pred_mean + y_pred_std

fig_pred = go.Figure()

# Región de confianza
fig_pred.add_trace(go.Scatter(
    x=np.concatenate([idx, idx[::-1]]),
    y=np.concatenate([upper, lower[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo='skip',
    name='Media ± desviación'
))

# Media predictiva
fig_pred.add_trace(go.Scatter(
    x=idx,
    y=y_pred_mean,
    mode='lines',
    name='Media predicha',
    line=dict(color='blue', width=2)
))

# Valores reales como scatter
fig_pred.add_trace(go.Scatter(
    x=idx,
    y=y_test,
    mode='markers',
    name='Potencia real',
    marker=dict(color='black', size=7, symbol='star')
))

fig_pred.update_layout(
    title='Predicción bayesiana sobre datos de Colombia',
    xaxis_title='Número de muestra',
    yaxis_title='Potencia [kW]',
    template='plotly_white',
    hovermode='x unified',
    width=1100,
    height=650
)

fig_pred.show()

In [ ]:
fig_hist = go.Figure()

fig_hist.add_trace(go.Histogram(
    x=y_pred_mean,
    nbinsx=40,
    name='Potencias estimadas'
))

fig_hist.update_layout(
    title='Distribución de las potencias estimadas por el regresor',
    xaxis_title='Potencia estimada [kW]',
    yaxis_title='Frecuencia',
    template='plotly_white'
)

fig_hist.show()

In [ ]:
# Predicción sobre train
y_train_mean, y_train_std = model.predict(X_train_scaled, return_std=True)

train_idx = np.arange(len(y_train))
train_lower = y_train_mean - y_train_std
train_upper = y_train_mean + y_train_std

fig_train_pred = go.Figure()

# Región de confianza
fig_train_pred.add_trace(go.Scatter(
    x=np.concatenate([train_idx, train_idx[::-1]]),
    y=np.concatenate([train_upper, train_lower[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo='skip',
    name='Región de confianza'
))

# Media predictiva
fig_train_pred.add_trace(go.Scatter(
    x=train_idx,
    y=y_train_mean,
    mode='lines',
    name='Media predicha',
    line=dict(color='blue', width=2)
))

# Potencia real de train
fig_train_pred.add_trace(go.Scatter(
    x=train_idx,
    y=y_train,
    mode='markers',
    name='Potencia real train',
    marker=dict(color='black', size=7, symbol='star')
))

fig_train_pred.update_layout(
    title='Predicción bayesiana sobre conjunto de entrenamiento',
    xaxis_title='Número de muestra',
    yaxis_title='Potencia [kW]',
    template='plotly_white',
    hovermode='x unified',
    width=1100,
    height=650
)

fig_train_pred.show()

In [ ]:
fig_hist_train_pred = go.Figure()

fig_hist_train_pred.add_trace(go.Histogram(
    x=y_train_mean,
    nbinsx=40,
    name='Potencias estimadas train'
))

fig_hist_train_pred.update_layout(
    title='Distribución de las potencias estimadas en train',
    xaxis_title='Potencia estimada [kW]',
    yaxis_title='Frecuencia',
    template='plotly_white',
    width=1100,
    height=650
)

fig_hist_train_pred.show()

fig_hist_train = go.Figure()

In [ ]:
fig_hist_train_pred = go.Figure()

fig_hist_train_pred.add_trace(go.Histogram(
    x=y_train,
    nbinsx=40,
    name='Potencias estimadas train'
))

fig_hist_train_pred.update_layout(
    title='Distribución de las potencias reales en train',
    xaxis_title='Potencia estimada [kW]',
    yaxis_title='Frecuencia',
    template='plotly_white',
    width=1100,
    height=650
)

fig_hist_train_pred.show()

fig_hist_train = go.Figure()

# **2. Modelo con cross validation y resultados con mejor fold**

In [ ]:
# Datos Usa
X_cv = df[[
    "Geothermal Header Hot Water Temperature",
    "Ambient Temperature",
    "Hot Water Supply Flow Rate"
]].to_numpy()

y_cv = df[TARGET_COL_USA].to_numpy()

# Datos Colombia para test externo
X_col_test = df_filtrado[[
    "T____C_",
    "TEMPERATURA_AMBIENTE___C_",
    "Potencial_Agua_promd_gpm"
]].to_numpy()

y_col_test = df_filtrado[TARGET_COL_COL].to_numpy()

# K-FOLD
kf = KFold(n_splits=5, shuffle=True, random_state=GLOBAL_RANDOM_SEED)

y_oof_mean = np.zeros(len(y_cv))
y_oof_std = np.zeros(len(y_cv))

mae_folds = []
rmse_folds = []
r2_folds = []

# Para guardar el mejor fold
best_fold = None
best_r2 = -np.inf
best_model = None
best_scaler = None
best_val_idx = None
best_y_val = None
best_y_val_mean = None
best_y_val_std = None

In [ ]:
# 4) Cross-validation
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_cv), start=1):
    X_tr, X_val = X_cv[tr_idx], X_cv[val_idx]
    y_tr, y_val = y_cv[tr_idx], y_cv[val_idx]

    scaler_cv = StandardScaler()
    X_tr_scaled = scaler_cv.fit_transform(X_tr)
    X_val_scaled = scaler_cv.transform(X_val)

    model_cv = BayesianRidge(compute_score=True, max_iter=3000)
    model_cv.fit(X_tr_scaled, y_tr)

    y_val_mean, y_val_std = model_cv.predict(X_val_scaled, return_std=True)

    y_oof_mean[val_idx] = y_val_mean
    y_oof_std[val_idx] = y_val_std

    mae_fold = mean_absolute_error(y_val, y_val_mean)
    rmse_fold = np.sqrt(mean_squared_error(y_val, y_val_mean))
    r2_fold = r2_score(y_val, y_val_mean)

    mae_folds.append(mae_fold)
    rmse_folds.append(rmse_fold)
    r2_folds.append(r2_fold)

    print(f"Fold {fold}: MAE={mae_fold:.4f}, RMSE={rmse_fold:.4f}, R²={r2_fold:.4f}")

    # Guardar el mejor fold según R2
    if r2_fold > best_r2:
        best_r2 = r2_fold
        best_fold = fold
        best_model = copy.deepcopy(model_cv)
        best_scaler = copy.deepcopy(scaler_cv)
        best_val_idx = val_idx.copy()
        best_y_val = y_val.copy()
        best_y_val_mean = y_val_mean.copy()
        best_y_val_std = y_val_std.copy()

print("\nResultados")
print(f"MAE  = {np.mean(mae_folds):.4f} ± {np.std(mae_folds):.4f}")
print(f"RMSE = {np.mean(rmse_folds):.4f} ± {np.std(rmse_folds):.4f}")
print(f"R²   = {np.mean(r2_folds):.4f} ± {np.std(r2_folds):.4f}")

print(f"\nMejor fold: {best_fold}")
print(f"Mejor R² de validación: {best_r2:.4f}")

Fold 1: MAE=3.6710, RMSE=5.0958, R²=0.4021
Fold 2: MAE=4.2458, RMSE=5.8806, R²=0.4232
Fold 3: MAE=4.2865, RMSE=5.9173, R²=0.3321
Fold 4: MAE=4.5717, RMSE=6.3146, R²=0.3713
Fold 5: MAE=4.0201, RMSE=5.5172, R²=0.4624

Resultados
MAE  = 4.1590 ± 0.3005
RMSE = 5.7451 ± 0.4113
R²   = 0.3982 ± 0.0444

Mejor fold: 5
Mejor R² de validación: 0.4624


In [ ]:
# Gráfica del mejor fold en validación
best_idx_plot = np.arange(len(best_y_val))
best_lower = best_y_val_mean - best_y_val_std
best_upper = best_y_val_mean + best_y_val_std

fig_best_fold = go.Figure()

fig_best_fold.add_trace(go.Scatter(
    x=np.concatenate([best_idx_plot, best_idx_plot[::-1]]),
    y=np.concatenate([best_upper, best_lower[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo='skip',
    name='Región de confianza'
))

fig_best_fold.add_trace(go.Scatter(
    x=best_idx_plot,
    y=best_y_val_mean,
    mode='lines',
    name='Media predictiva',
    line=dict(color='blue', width=2)
))

fig_best_fold.add_trace(go.Scatter(
    x=best_idx_plot,
    y=best_y_val,
    mode='markers',
    name='Potencia real fold',
    marker=dict(color='black', size=7, symbol='star')
))

fig_best_fold.update_layout(
    title=f'Mejor fold de validación cruzada (Fold {best_fold})',
    xaxis_title='Número de muestra',
    yaxis_title='Potencia [kW]',
    template='plotly_white',
    hovermode='x unified',
    width=1100,
    height=650
)

fig_best_fold.show()

# Test con datos de C   olombia y el mejor fold
X_col_scaled = best_scaler.transform(X_col_test)
y_col_mean, y_col_std = best_model.predict(X_col_scaled, return_std=True)

mae_col = mean_absolute_error(y_col_test, y_col_mean)
rmse_col = np.sqrt(mean_squared_error(y_col_test, y_col_mean))
r2_col = r2_score(y_col_test, y_col_mean)

print("\nResultados de test en Colombia con el mejor fold")
print(f"Fold usado = {best_fold}")
print(f"MAE  = {mae_col:.4f}")
print(f"RMSE = {rmse_col:.4f}")
print(f"R²   = {r2_col:.4f}")

#Grráfica datos Colombia
col_idx = np.arange(len(y_col_test))
col_lower = y_col_mean - y_col_std
col_upper = y_col_mean + y_col_std

fig_col = go.Figure()

fig_col.add_trace(go.Scatter(
    x=np.concatenate([col_idx, col_idx[::-1]]),
    y=np.concatenate([col_upper, col_lower[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo='skip',
    name='Región de confianza'
))

fig_col.add_trace(go.Scatter(
    x=col_idx,
    y=y_col_mean,
    mode='lines',
    name='Media predictiva',
    line=dict(color='blue', width=2)
))

fig_col.add_trace(go.Scatter(
    x=col_idx,
    y=y_col_test,
    mode='markers',
    name='Potencia real Colombia',
    marker=dict(color='black', size=7, symbol='star')
))

fig_col.update_layout(
    title=f'Test en Colombia con el mejor fold (Fold {best_fold})',
    xaxis_title='Número de muestra',
    yaxis_title='Potencia [kW]',
    template='plotly_white',
    hovermode='x unified',
    width=1100,
    height=650
)

fig_col.show()

# Histograma de las predicciones de Colombia
fig_hist_col = go.Figure()

fig_hist_col.add_trace(go.Histogram(
    x=y_col_mean,
    nbinsx=40,
    name='Potencias estimadas Colombia',
    marker=dict(color='blue')
))

fig_hist_col.update_layout(
    title='Distribución de potencias estimadas en Colombia',
    xaxis_title='Potencia estimada [kW]',
    yaxis_title='Frecuencia',
    template='plotly_white',
    width=1100,
    height=650
)

fig_hist_col.show()


Resultados de test en Colombia con el mejor fold
Fold usado = 5
MAE  = 28.6076
RMSE = 30.3483
R²   = -21.5235


# **Cross validation con los dos conjuntos de datos juntos**

In [ ]:
# Datos USA
X_usa = df[[
    "Geothermal Header Hot Water Temperature",
    "Ambient Temperature",
    "Hot Water Supply Flow Rate"
]].to_numpy()

y_usa = df[TARGET_COL_USA].to_numpy()

# Datos Colombia
X_col = df_filtrado[[
    "T____C_",
    "TEMPERATURA_AMBIENTE___C_",
    "Potencial_Agua_promd_gpm"
]].to_numpy()

y_col = df_filtrado[TARGET_COL_COL].to_numpy()

X_all = np.vstack([X_usa, X_col])
y_all = np.concatenate([y_usa, y_col])

# Número de muestras
source_all = np.array(
    ["USA"] * len(y_usa) + ["COL"] * len(y_col)
)

print("Tamaño USA:", X_usa.shape, y_usa.shape)
print("Tamaño COL:", X_col.shape, y_col.shape)
print("Tamaño combinado:", X_all.shape, y_all.shape)

# 4) K-fold sobre el conjunto combinado
kf = KFold(n_splits=5, shuffle=True, random_state=GLOBAL_RANDOM_SEED)

y_oof_mean = np.zeros(len(y_all))
y_oof_std = np.zeros(len(y_all))

mae_folds = []
rmse_folds = []
r2_folds = []

# Cross validation
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_all), start=1):
    X_tr, X_val = X_all[tr_idx], X_all[val_idx]
    y_tr, y_val = y_all[tr_idx], y_all[val_idx]

    scaler_cv = StandardScaler()
    X_tr_scaled = scaler_cv.fit_transform(X_tr)
    X_val_scaled = scaler_cv.transform(X_val)

    model_cv = BayesianRidge(compute_score=True, max_iter=2000)
    model_cv.fit(X_tr_scaled, y_tr)

    y_val_mean, y_val_std = model_cv.predict(X_val_scaled, return_std=True)

    y_oof_mean[val_idx] = y_val_mean
    y_oof_std[val_idx] = y_val_std

    mae_fold = mean_absolute_error(y_val, y_val_mean)
    rmse_fold = np.sqrt(mean_squared_error(y_val, y_val_mean))
    r2_fold = r2_score(y_val, y_val_mean)

    mae_folds.append(mae_fold)
    rmse_folds.append(rmse_fold)
    r2_folds.append(r2_fold)

    n_usa = np.sum(source_all[val_idx] == "USA")
    n_col = np.sum(source_all[val_idx] == "COL")

    print(
        f"Fold {fold}: "
        f"MAE={mae_fold:.4f}, RMSE={rmse_fold:.4f}, R²={r2_fold:.4f} "
        f"| val USA={n_usa}, val COL={n_col}"
    )

print(f"MAE  = {np.mean(mae_folds):.4f} ± {np.std(mae_folds):.4f}")
print(f"RMSE = {np.mean(rmse_folds):.4f} ± {np.std(rmse_folds):.4f}")
print(f"R²   = {np.mean(r2_folds):.4f} ± {np.std(r2_folds):.4f}")

Tamaño USA: (1311, 3) (1311,)
Tamaño COL: (569, 3) (569,)
Tamaño combinado: (1880, 3) (1880,)
Fold 1: MAE=6.3139, RMSE=10.1627, R²=0.7682 | val USA=250, val COL=126
Fold 2: MAE=6.2333, RMSE=8.4412, R²=0.8344 | val USA=264, val COL=112
Fold 3: MAE=5.8022, RMSE=7.7262, R²=0.8572 | val USA=272, val COL=104
Fold 4: MAE=6.1236, RMSE=8.4140, R²=0.8299 | val USA=278, val COL=98
Fold 5: MAE=5.9031, RMSE=8.0531, R²=0.8558 | val USA=247, val COL=129
MAE  = 6.0752 ± 0.1942
RMSE = 8.5595 ± 0.8434
R²   = 0.8291 ± 0.0324


In [ ]:
idx_all = np.arange(len(y_all))
lower_all = y_oof_mean - y_oof_std
upper_all = y_oof_mean + y_oof_std

fig_cv = go.Figure()

fig_cv.add_trace(go.Scatter(
    x=np.concatenate([idx_all, idx_all[::-1]]),
    y=np.concatenate([upper_all, lower_all[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo='skip',
    name='Región de confianza'
))

fig_cv.add_trace(go.Scatter(
    x=idx_all,
    y=y_oof_mean,
    mode='lines',
    name='Media predictiva CV',
    line=dict(color='blue', width=2)
))

# Puntos reales USA
mask_usa = source_all == "USA"
fig_cv.add_trace(go.Scatter(
    x=idx_all[mask_usa],
    y=y_all[mask_usa],
    mode='markers',
    name='Potencia real USA',
    marker=dict(color='black', size=7, symbol='star')
))

# Puntos reales Colombia
mask_col = source_all == "COL"
fig_cv.add_trace(go.Scatter(
    x=idx_all[mask_col],
    y=y_all[mask_col],
    mode='markers',
    name='Potencia real Colombia',
    marker=dict(color='gray', size=6, symbol='circle')
))

fig_cv.update_layout(
    title='Predicción out-of-fold con validación cruzada sobre dataset combinado',
    xaxis_title='Número de muestra',
    yaxis_title='Potencia [kW]',
    template='plotly_white',
    hovermode='x unified',
    width=1100,
    height=650
)

fig_cv.show()

# Histograma real vs predicho con valores OOF
fig_compare = go.Figure()

fig_compare.add_trace(go.Histogram(
    x=y_all,
    nbinsx=20,
    name='Potencia real combinada',
    opacity=0.50,
    marker=dict(color='black')
))

fig_compare.add_trace(go.Histogram(
    x=y_oof_mean,
    nbinsx=20,
    name='Potencia predicha CV',
    opacity=0.50,
    marker=dict(color='blue')
))

fig_compare.update_layout(
    title='Comparación de distribuciones: real vs predicha',
    xaxis_title='Potencia [kW]',
    yaxis_title='Frecuencia',
    template='plotly_white',
    barmode='overlay',
    width=1100,
    height=650
)

fig_compare.show()

# **Regresor bayesiano con base de datos concatenada**

In [ ]:
from sklearn.model_selection import train_test_split

scaler = StandardScaler()

x_train, x_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=GLOBAL_RANDOM_SEED
)

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

In [ ]:
model_2 = BayesianRidge(compute_score=True, max_iter=3000)
model_2.fit(x_train_scaled, y_train)

y_pred_mean, y_pred_std = model_2.predict(x_test_scaled, return_std=True)

In [ ]:
mae = mean_absolute_error(y_test, y_pred_mean)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_mean))
r2 = r2_score(y_test, y_pred_mean)

print(f"MAE test = {mae:.4f}")
print(f"RMSE test= {rmse:.4f}")
print(f"R² test = {r2:.4f}")

MAE test = 6.3139
RMSE test= 10.1627
R² test = 0.7682


In [ ]:
y_pred_mean_train, y_pred_std_train = model_2.predict(x_train_scaled, return_std=True)
mae = mean_absolute_error(y_train, y_pred_mean_train)
rmse = np.sqrt(mean_squared_error(y_train, y_pred_mean_train))
r2 = r2_score(y_train, y_pred_mean_train)

print(f"MAE train = {mae:.4f}")
print(f"RMSE train = {rmse:.4f}")
print(f"R² train = {r2:.4f}")

MAE train = 5.7257
RMSE train = 8.0826
R² train = 0.8484


In [ ]:
idx = np.arange(len(y_test))
lower = y_pred_mean - y_pred_std
upper = y_pred_mean + y_pred_std

fig_pred = go.Figure()

# Región de confianza
fig_pred.add_trace(go.Scatter(
    x=np.concatenate([idx, idx[::-1]]),
    y=np.concatenate([upper, lower[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo='skip',
    name='Media ± desviación'
))

# Media predictiva
fig_pred.add_trace(go.Scatter(
    x=idx,
    y=y_pred_mean,
    mode='lines',
    name='Media predicha',
    line=dict(color='blue', width=2)
))

# Valores reales como scatter
fig_pred.add_trace(go.Scatter(
    x=idx,
    y=y_test,
    mode='markers',
    name='Potencia real',
    marker=dict(color='black', size=7, symbol='star')
))

fig_pred.update_layout(
    title='Predicción bayesiana sobre datos de Colombia',
    xaxis_title='Número de muestra',
    yaxis_title='Potencia [kW]',
    template='plotly_white',
    hovermode='x unified',
    width=1100,
    height=650
)

fig_pred.show()

In [ ]:
fig_hist = go.Figure()

fig_hist.add_trace(go.Histogram(
    x=y_pred_mean,
    nbinsx=40,
    name='Potencias estimadas'
))

fig_hist.update_layout(
    title='Distribución de las potencias estimadas por el regresor',
    xaxis_title='Potencia estimada [kW]',
    yaxis_title='Frecuencia',
    template='plotly_white'
)

fig_hist.show()

In [ ]:
# Predicción sobre train
y_train_mean, y_train_std = model.predict(x_train_scaled, return_std=True)

train_idx = np.arange(len(y_train))
train_lower = y_train_mean - y_train_std
train_upper = y_train_mean + y_train_std

fig_train_pred = go.Figure()

# Región de confianza
fig_train_pred.add_trace(go.Scatter(
    x=np.concatenate([train_idx, train_idx[::-1]]),
    y=np.concatenate([train_upper, train_lower[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo='skip',
    name='Región de confianza'
))

# Media predictiva
fig_train_pred.add_trace(go.Scatter(
    x=train_idx,
    y=y_train_mean,
    mode='lines',
    name='Media predicha',
    line=dict(color='blue', width=2)
))

# Potencia real de train
fig_train_pred.add_trace(go.Scatter(
    x=train_idx,
    y=y_train,
    mode='markers',
    name='Potencia real train',
    marker=dict(color='black', size=7, symbol='star')
))

fig_train_pred.update_layout(
    title='Predicción bayesiana sobre conjunto de entrenamiento',
    xaxis_title='Número de muestra',
    yaxis_title='Potencia [kW]',
    template='plotly_white',
    hovermode='x unified',
    width=1100,
    height=650
)

fig_train_pred.show()

#Nuevo


In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 1) Construcción de datos
# =========================

# Datos USA
X_usa = df[[
    "Geothermal Header Hot Water Temperature",
    "Ambient Temperature",
    "Hot Water Supply Flow Rate"
]].to_numpy()

y_usa = df[TARGET_COL_USA].to_numpy()

# Datos Colombia
X_col = df_filtrado[[
    "T____C_",
    "TEMPERATURA_AMBIENTE___C_",
    "Potencial_Agua_promd_gpm"
]].to_numpy()

y_col = df_filtrado[TARGET_COL_COL].to_numpy()

X_all = np.vstack([X_usa, X_col])
y_all = np.concatenate([y_usa, y_col])

source_all = np.array(
    ["USA"] * len(y_usa) + ["COL"] * len(y_col)
)

print("Tamaño USA:", X_usa.shape, y_usa.shape)
print("Tamaño COL:", X_col.shape, y_col.shape)
print("Tamaño combinado:", X_all.shape, y_all.shape)

# ============================================================
# 2) Crear folds base
#    Si quieres EXACTAMENTE como la imagen, usa 6 folds
#    Si quieres 5 folds, cambia n_folds = 5
# ============================================================

n_folds = 6

kf = KFold(
    n_splits=n_folds,
    shuffle=True,              # deja True si quieres folds aleatorios
    random_state=GLOBAL_RANDOM_SEED
)

# fold_id[i] = número de fold al que pertenece la muestra i
fold_id = np.empty(len(y_all), dtype=int)

for f, (_, idx_fold) in enumerate(kf.split(X_all)):
    fold_id[idx_fold] = f

# ============================================================
# 3) Estructura tipo imagen:
#    En cada iteración:
#      - 1 fold = TEST
#      - 1 fold = VAL
#      - resto = TRAIN
#
#    Aquí usamos una rotación:
#      test_fold = f
#      val_fold  = (f + 1) % n_folds
# ============================================================

# Predicciones OOF sobre TEST (cada muestra será test una sola vez)
y_oof_test_mean = np.zeros(len(y_all))
y_oof_test_std = np.zeros(len(y_all))

# Opcional: guardar también predicciones sobre VALIDACIÓN
y_oof_val_mean = np.full(len(y_all), np.nan)
y_oof_val_std = np.full(len(y_all), np.nan)

mae_val_folds = []
rmse_val_folds = []
r2_val_folds = []

mae_test_folds = []
rmse_test_folds = []
r2_test_folds = []

for test_fold in range(n_folds):
    val_fold = (test_fold + 1) % n_folds

    train_mask = (fold_id != test_fold) & (fold_id != val_fold)
    val_mask   = (fold_id == val_fold)
    test_mask  = (fold_id == test_fold)

    X_tr, y_tr = X_all[train_mask], y_all[train_mask]
    X_val, y_val = X_all[val_mask], y_all[val_mask]
    X_test, y_test = X_all[test_mask], y_all[test_mask]

    # Escalado SOLO con entrenamiento
    scaler_cv = StandardScaler()
    X_tr_scaled = scaler_cv.fit_transform(X_tr)
    X_val_scaled = scaler_cv.transform(X_val)
    X_test_scaled = scaler_cv.transform(X_test)

    # Modelo
    model_cv = BayesianRidge(compute_score=True, max_iter=2000)
    model_cv.fit(X_tr_scaled, y_tr)

    # Predicción en validación
    y_val_mean, y_val_std = model_cv.predict(X_val_scaled, return_std=True)

    # Predicción en test
    y_test_mean, y_test_std = model_cv.predict(X_test_scaled, return_std=True)

    # Guardar OOF
    y_oof_val_mean[val_mask] = y_val_mean
    y_oof_val_std[val_mask] = y_val_std

    y_oof_test_mean[test_mask] = y_test_mean
    y_oof_test_std[test_mask] = y_test_std

    # Métricas validación
    mae_val = mean_absolute_error(y_val, y_val_mean)
    rmse_val = np.sqrt(mean_squared_error(y_val, y_val_mean))
    r2_val = r2_score(y_val, y_val_mean)

    mae_val_folds.append(mae_val)
    rmse_val_folds.append(rmse_val)
    r2_val_folds.append(r2_val)

    # Métricas test
    mae_test = mean_absolute_error(y_test, y_test_mean)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_mean))
    r2_test = r2_score(y_test, y_test_mean)

    mae_test_folds.append(mae_test)
    rmse_test_folds.append(rmse_test)
    r2_test_folds.append(r2_test)

    # Conteos por fuente
    n_train_usa = np.sum(source_all[train_mask] == "USA")
    n_train_col = np.sum(source_all[train_mask] == "COL")

    n_val_usa = np.sum(source_all[val_mask] == "USA")
    n_val_col = np.sum(source_all[val_mask] == "COL")

    n_test_usa = np.sum(source_all[test_mask] == "USA")
    n_test_col = np.sum(source_all[test_mask] == "COL")

    print(f"\nFold {test_fold + 1}/{n_folds}")
    print(f"  TRAIN -> USA={n_train_usa}, COL={n_train_col}, total={train_mask.sum()}")
    print(f"  VAL   -> USA={n_val_usa}, COL={n_val_col}, total={val_mask.sum()}")
    print(f"  TEST  -> USA={n_test_usa}, COL={n_test_col}, total={test_mask.sum()}")

    print(
        f"  VAL  : MAE={mae_val:.4f}, RMSE={rmse_val:.4f}, R²={r2_val:.4f}"
    )
    print(
        f"  TEST : MAE={mae_test:.4f}, RMSE={rmse_test:.4f}, R²={r2_test:.4f}"
    )

# =========================
# 4) Resumen final
# =========================

print("\n===== PROMEDIO EN VALIDACIÓN =====")
print(f"MAE  = {np.mean(mae_val_folds):.4f} ± {np.std(mae_val_folds):.4f}")
print(f"RMSE = {np.mean(rmse_val_folds):.4f} ± {np.std(rmse_val_folds):.4f}")
print(f"R²   = {np.mean(r2_val_folds):.4f} ± {np.std(r2_val_folds):.4f}")

print("\n===== PROMEDIO EN TEST =====")
print(f"MAE  = {np.mean(mae_test_folds):.4f} ± {np.std(mae_test_folds):.4f}")
print(f"RMSE = {np.mean(rmse_test_folds):.4f} ± {np.std(rmse_test_folds):.4f}")
print(f"R²   = {np.mean(r2_test_folds):.4f} ± {np.std(r2_test_folds):.4f}")

# Métrica global OOF sobre TEST
mae_oof_test = mean_absolute_error(y_all, y_oof_test_mean)
rmse_oof_test = np.sqrt(mean_squared_error(y_all, y_oof_test_mean))
r2_oof_test = r2_score(y_all, y_oof_test_mean)

print("\n===== OOF GLOBAL EN TEST =====")
print(f"MAE  = {mae_oof_test:.4f}")
print(f"RMSE = {rmse_oof_test:.4f}")
print(f"R²   = {r2_oof_test:.4f}")

Tamaño USA: (1311, 3) (1311,)
Tamaño COL: (569, 3) (569,)
Tamaño combinado: (1880, 3) (1880,)

Fold 1/6
  TRAIN -> USA=887, COL=365, total=1252
  VAL   -> USA=215, COL=99, total=314
  TEST  -> USA=209, COL=105, total=314
  VAL  : MAE=6.1332, RMSE=9.2373, R²=0.8073
  TEST : MAE=6.1760, RMSE=9.8328, R²=0.7808

Fold 2/6
  TRAIN -> USA=874, COL=379, total=1253
  VAL   -> USA=222, COL=91, total=313
  TEST  -> USA=215, COL=99, total=314
  VAL  : MAE=5.8743, RMSE=7.6301, R²=0.8666
  TEST : MAE=6.4438, RMSE=9.1835, R²=0.8095

Fold 3/6
  TRAIN -> USA=853, COL=401, total=1254
  VAL   -> USA=236, COL=77, total=313
  TEST  -> USA=222, COL=91, total=313
  VAL  : MAE=6.1243, RMSE=8.3803, R²=0.8163
  TEST : MAE=5.8697, RMSE=7.5766, R²=0.8684

Fold 4/6
  TRAIN -> USA=856, COL=398, total=1254
  VAL   -> USA=219, COL=94, total=313
  TEST  -> USA=236, COL=77, total=313
  VAL  : MAE=6.0167, RMSE=8.1409, R²=0.8542
  TEST : MAE=6.0796, RMSE=8.3965, R²=0.8155

Fold 5/6
  TRAIN -> USA=882, COL=372, total=1254